In [ ]:
import pandas as pd
import evaluate
import numpy as np
from tabulate import tabulate
import time # Import time for basic timing if needed, though evaluate handles internal timing

# --- Load evaluation metrics ---
print("Loading evaluation metrics...")
start_time = time.time()
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
# Specify model type for BERTScore for potential speedup/correctness if needed, though default usually works
# For example: bertscore_metric = evaluate.load("bertscore", model_type="distilbert-base-uncased")
bertscore_metric = evaluate.load("bertscore")
print(f"Metrics loaded in {time.time() - start_time:.2f} seconds.")
# --- End loading metrics ---

def evaluate_model(df, model_name):
    """
    Evaluate a model's predictions using BLEU, ROUGE-L, and BERTScore.
    Also compute average length of generated jokes.
    """
    print(f"  Evaluating {len(df)} samples for model '{model_name}'...") # Added verbosity
    setups = df['setup'].tolist()
    gt_punchlines = df['ground_truth_punchline'].tolist()
    # Handle potential missing values or type issues gracefully
    generated = df['generated_punchline'].fillna('').astype(str).apply(lambda x: x.split('\n')[0]).tolist()
    # generated = df['generated_punchline'].fillna('').astype(str).tolist()

    references = [f"{i} {j}" for i, j in zip(setups, gt_punchlines)]
    predictions = generated
    
    # Calculate average length of generated content
    # For Mistral models: only punchline length
    # For other models: full generated content length (setup + punchline)
    if "Mistral Zero-Shot" in model_name or 'Mistral Finetuned' in model_name:
        print("  Applying different evaluation (including setup in prediction).") # Added verbosity
        predictions = [f"{i} {j}" for i, j in zip(setups, generated)]
        # For Mistral models, average length is just the punchline
        avg_length = np.mean([len(g.split()) for g in generated])
    else:
        # For other models, generated_punchline contains setup+punchline, so measure full length
        avg_length = np.mean([len(g.split()) for g in generated])

    # --- Compute Metrics ---
    print("    Computing BLEU...") # Added verbosity
    bleu_res = bleu_metric.compute(predictions=predictions, references=references)
    print("    Computing ROUGE...") # Added verbosity
    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    print("    Computing BERTScore...") # Added verbosity
    # Consider adding batch_size parameter if memory issues arise or for potential speedup
    bert_res = bertscore_metric.compute(predictions=predictions, references=references, lang="en", batch_size=128, verbose=True)
    # bert_res = bertscore_metric.compute(predictions=predictions, references=references, lang="en")
    print("    Evaluation complete for this batch.") # Added verbosity
    # --- End Compute Metrics ---

    return {
        'bleu': bleu_res['bleu'],
        'rougeL': rouge_res['rougeL'],
        'bertscore': np.mean(bert_res['f1']), # Ensure we get a single mean value
        'avg_length': avg_length
    }


def process_seeds(seed_files, model_name):
    """
    Process and compute metrics for all seeds of a model.
    """
    metrics = {'bleu': [], 'rougeL': [], 'bertscore': [], 'avg_length': []}
    num_seeds = len(seed_files)

    print(f"Processing {num_seeds} seed files for model '{model_name}':") # Added verbosity
    for i, seed_file in enumerate(seed_files):
        print(f"  Processing seed file {i+1}/{num_seeds}: {seed_file}") # Added verbosity
        try:
            df = pd.read_csv(seed_file)
            # Basic check for required columns
            required_cols = ['setup', 'ground_truth_punchline', 'generated_punchline']
            if not all(col in df.columns for col in required_cols):
                print(f"    Warning: Skipping file {seed_file} due to missing columns. Required: {required_cols}")
                continue # Skip this file if columns are missing

            # Pass the specific model name for potential logic inside evaluate_model
            res = evaluate_model(df, model_name)
            metrics['bleu'].append(res['bleu'])
            metrics['rougeL'].append(res['rougeL'])
            metrics['bertscore'].append(res['bertscore'])
            metrics['avg_length'].append(res['avg_length'])
            print(f"    Finished processing {seed_file}.") # Added verbosity

        except FileNotFoundError:
            print(f"    Warning: Seed file not found: {seed_file}. Skipping.") # Added warning
        except Exception as e:
            print(f"    Warning: Error processing file {seed_file}: {e}. Skipping.") # Added generic error handling

    if not metrics['bleu']: # Check if any seeds were successfully processed
         print(f"  Warning: No valid data processed for model '{model_name}'. Returning zero stats.")
         stats = {
             'bleu_mean': 0.0, 'bleu_std': 0.0,
             'rougeL_mean': 0.0, 'rougeL_std': 0.0,
             'bertscore_mean': 0.0, 'bertscore_std': 0.0,
             'avg_length_mean': 0.0, 'avg_length_std': 0.0,
         }
    else:
        # Calculate stats only if there's data
        print(f"  Calculating statistics for model '{model_name}'...") # Added verbosity
        stats = {
            'bleu_mean': np.mean(metrics['bleu']),
            'bleu_std': np.std(metrics['bleu']),
            'rougeL_mean': np.mean(metrics['rougeL']),
            'rougeL_std': np.std(metrics['rougeL']),
            'bertscore_mean': np.mean(metrics['bertscore']),
            'bertscore_std': np.std(metrics['bertscore']),
            'avg_length_mean': np.mean(metrics['avg_length']),
            'avg_length_std': np.std(metrics['avg_length']),
        }
        print(f"  Statistics calculated for {model_name}.") # Added verbosity

    return stats


# Define seed files for each model
# Make sure these paths are correct relative to where you run the script
models = {
    'Finetuned GPT-2': [f"../results/filtered_finetuned_gpt2_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD Imitation+DPO': [f"../results/filtered_gpt2_acl_dpo_seed_{i}.csv" for i in range(5)],
    'Mistral Zero-Shot': [f"../results/filtered_zero_shot_mistral_seed_{i}.csv" for i in range(5)],
    'Mistral Finetuned': [f"../results/filtered_finetuned_mistral_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Finetuned Mistral)': [f"../results/filtered_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(5)],
    'GPT-2 Mistral (LoRA)': [f"../results/filtered_gpt2_mistral_lora_seed_{i}.csv" for i in range(5)],
    'GPT-2 Mistral (Frozen)': [f"../results/filtered_frozen_mistral_seed_{i}.csv" for i in range(5)], #this is gpt2_frozen_mistral
    'GPT-2 KD (Imitation)': [f"../results/filtered_gpt2_imitation_seed_{i}.csv" for i in range(5)],
}


all_results = []
print("\n--- Starting Evaluation Loop for All Models ---") # Added verbosity
total_models = len(models)

for i, (name, files) in enumerate(models.items()):
    print(f"\n--- Processing Model {i+1}/{total_models}: {name} ---") # Added verbosity
    start_model_time = time.time()
    stats = process_seeds(files, name)
    all_results.append({
        'Model': name,
        'BLEU': f"{stats['bleu_mean']:.4f} ± {stats['bleu_std']:.4f}",
        'ROUGE-L': f"{stats['rougeL_mean']:.4f} ± {stats['rougeL_std']:.4f}",
        'BERTScore': f"{stats['bertscore_mean']:.4f} ± {stats['bertscore_std']:.4f}",
        'Avg Length': f"{stats['avg_length_mean']:.1f} ± {stats['avg_length_std']:.1f}",
    })
    end_model_time = time.time()
    print(f"--- Finished Model {name} in {end_model_time - start_model_time:.2f} seconds ---") # Added verbosity

print("\n--- All Models Processed. Generating Final Results Table ---") # Added verbosity

# Check if any results were generated
if not all_results:
    print("No results were generated. Exiting.")
else:
    # Print the final table
    print(tabulate(all_results, headers="keys", tablefmt="grid"))

print("\n--- Evaluation Script Finished ---") # Added verbosity

In [ ]:
import pandas as pd
import evaluate
import numpy as np
from tabulate import tabulate
import time # Import time for basic timing if needed, though evaluate handles internal timing

# --- Load evaluation metrics ---
print("Loading evaluation metrics...")
start_time = time.time()
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
# Specify model type for BERTScore for potential speedup/correctness if needed, though default usually works
# For example: bertscore_metric = evaluate.load("bertscore", model_type="distilbert-base-uncased")
bertscore_metric = evaluate.load("bertscore")
print(f"Metrics loaded in {time.time() - start_time:.2f} seconds.")
# --- End loading metrics ---

def evaluate_model(df, model_name):
    """
    Evaluate a model's predictions using BLEU, ROUGE-L, and BERTScore.
    Also compute average length of generated jokes.
    """
    print(f"  Evaluating {len(df)} samples for model '{model_name}'...") # Added verbosity
    setups = df['setup'].tolist()
    gt_punchlines = df['ground_truth_punchline'].tolist()
    # Handle potential missing values or type issues gracefully
    # generated = df['generated_punchline'].fillna('').astype(str).apply(lambda x: x.split('\n')[0]).tolist()
    generated = df['generated_punchline'].fillna('').astype(str).apply(lambda x: ' '.join(x.split()[:15])).tolist()
    # generated = df['generated_punchline'].fillna('').astype(str).tolist()

    references = [f"{i} {j}" for i, j in zip(setups, gt_punchlines)]
    predictions = generated
    
    # Calculate average length of generated content
    # For Mistral models: only punchline length
    # For other models: full generated content length (setup + punchline)
    if "Mistral Zero-Shot" in model_name or 'Mistral Finetuned' in model_name:
        print("  Applying different evaluation (including setup in prediction).") # Added verbosity
        predictions = [f"{i} {j}" for i, j in zip(setups, generated)]
        # For Mistral models, average length is just the punchline
        avg_length = np.mean([len(g.split()) for g in generated])
    else:
        # For other models, generated_punchline contains setup+punchline, so measure full length
        avg_length = np.mean([len(g.split()) for g in generated])

    # --- Compute Metrics ---
    print("    Computing BLEU...") # Added verbosity
    bleu_res = bleu_metric.compute(predictions=predictions, references=references)
    print("    Computing ROUGE...") # Added verbosity
    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    print("    Computing BERTScore...") # Added verbosity
    # Consider adding batch_size parameter if memory issues arise or for potential speedup
    bert_res = bertscore_metric.compute(predictions=predictions, references=references, lang="en", batch_size=128, verbose=True)
    # bert_res = bertscore_metric.compute(predictions=predictions, references=references, lang="en")
    print("    Evaluation complete for this batch.") # Added verbosity
    # --- End Compute Metrics ---

    return {
        'bleu': bleu_res['bleu'],
        'rougeL': rouge_res['rougeL'],
        'bertscore': np.mean(bert_res['f1']), # Ensure we get a single mean value
        'avg_length': avg_length
    }


def process_seeds(seed_files, model_name):
    """
    Process and compute metrics for all seeds of a model.
    """
    metrics = {'bleu': [], 'rougeL': [], 'bertscore': [], 'avg_length': []}
    num_seeds = len(seed_files)

    print(f"Processing {num_seeds} seed files for model '{model_name}':") # Added verbosity
    for i, seed_file in enumerate(seed_files):
        print(f"  Processing seed file {i+1}/{num_seeds}: {seed_file}") # Added verbosity
        try:
            df = pd.read_csv(seed_file)
            # Basic check for required columns
            required_cols = ['setup', 'ground_truth_punchline', 'generated_punchline']
            if not all(col in df.columns for col in required_cols):
                print(f"    Warning: Skipping file {seed_file} due to missing columns. Required: {required_cols}")
                continue # Skip this file if columns are missing

            # Pass the specific model name for potential logic inside evaluate_model
            res = evaluate_model(df, model_name)
            metrics['bleu'].append(res['bleu'])
            metrics['rougeL'].append(res['rougeL'])
            metrics['bertscore'].append(res['bertscore'])
            metrics['avg_length'].append(res['avg_length'])
            print(f"    Finished processing {seed_file}.") # Added verbosity

        except FileNotFoundError:
            print(f"    Warning: Seed file not found: {seed_file}. Skipping.") # Added warning
        except Exception as e:
            print(f"    Warning: Error processing file {seed_file}: {e}. Skipping.") # Added generic error handling

    if not metrics['bleu']: # Check if any seeds were successfully processed
         print(f"  Warning: No valid data processed for model '{model_name}'. Returning zero stats.")
         stats = {
             'bleu_mean': 0.0, 'bleu_std': 0.0,
             'rougeL_mean': 0.0, 'rougeL_std': 0.0,
             'bertscore_mean': 0.0, 'bertscore_std': 0.0,
             'avg_length_mean': 0.0, 'avg_length_std': 0.0,
         }
    else:
        # Calculate stats only if there's data
        print(f"  Calculating statistics for model '{model_name}'...") # Added verbosity
        stats = {
            'bleu_mean': np.mean(metrics['bleu']),
            'bleu_std': np.std(metrics['bleu']),
            'rougeL_mean': np.mean(metrics['rougeL']),
            'rougeL_std': np.std(metrics['rougeL']),
            'bertscore_mean': np.mean(metrics['bertscore']),
            'bertscore_std': np.std(metrics['bertscore']),
            'avg_length_mean': np.mean(metrics['avg_length']),
            'avg_length_std': np.std(metrics['avg_length']),
        }
        print(f"  Statistics calculated for {model_name}.") # Added verbosity

    return stats


# Define seed files for each model
# Make sure these paths are correct relative to where you run the script
models = {
    'Finetuned GPT-2': [f"../results/filtered_finetuned_gpt2_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD Imitation+DPO': [f"../results/filtered_gpt2_acl_dpo_seed_{i}.csv" for i in range(5)],
    'Mistral Zero-Shot': [f"../results/filtered_zero_shot_mistral_seed_{i}.csv" for i in range(5)],
    'Mistral Finetuned': [f"../results/filtered_finetuned_mistral_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Finetuned Mistral)': [f"../results/filtered_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(5)],
    'GPT-2 Mistral (LoRA)': [f"../results/filtered_gpt2_mistral_lora_seed_{i}.csv" for i in range(5)],
    'GPT-2 Mistral (Frozen)': [f"../results/filtered_frozen_mistral_seed_{i}.csv" for i in range(5)], #this is gpt2_frozen_mistral
    'GPT-2 KD (Imitation)': [f"../results/filtered_gpt2_imitation_seed_{i}.csv" for i in range(5)],
}


all_results = []
print("\n--- Starting Evaluation Loop for All Models ---") # Added verbosity
total_models = len(models)

for i, (name, files) in enumerate(models.items()):
    print(f"\n--- Processing Model {i+1}/{total_models}: {name} ---") # Added verbosity
    start_model_time = time.time()
    stats = process_seeds(files, name)
    all_results.append({
        'Model': name,
        'BLEU': f"{stats['bleu_mean']:.4f} ± {stats['bleu_std']:.4f}",
        'ROUGE-L': f"{stats['rougeL_mean']:.4f} ± {stats['rougeL_std']:.4f}",
        'BERTScore': f"{stats['bertscore_mean']:.4f} ± {stats['bertscore_std']:.4f}",
        'Avg Length': f"{stats['avg_length_mean']:.1f} ± {stats['avg_length_std']:.1f}",
    })
    end_model_time = time.time()
    print(f"--- Finished Model {name} in {end_model_time - start_model_time:.2f} seconds ---") # Added verbosity

print("\n--- All Models Processed. Generating Final Results Table ---") # Added verbosity

# Check if any results were generated
if not all_results:
    print("No results were generated. Exiting.")
else:
    # Print the final table
    print(tabulate(all_results, headers="keys", tablefmt="grid"))

print("\n--- Evaluation Script Finished ---") # Added verbosity

In [ ]:
import pandas as pd
import evaluate
import numpy as np
from tabulate import tabulate
import time # Import time for basic timing if needed, though evaluate handles internal timing

# --- Load evaluation metrics ---
print("Loading evaluation metrics...")
start_time = time.time()
bleu_metric = evaluate.load("bleu")
rouge_metric = evaluate.load("rouge")
# Specify model type for BERTScore for potential speedup/correctness if needed, though default usually works
# For example: bertscore_metric = evaluate.load("bertscore", model_type="distilbert-base-uncased")
bertscore_metric = evaluate.load("bertscore")
print(f"Metrics loaded in {time.time() - start_time:.2f} seconds.")
# --- End loading metrics ---

def evaluate_model(df, model_name):
    """
    Evaluate a model's predictions using BLEU, ROUGE-L, and BERTScore.
    Also compute average length of generated jokes.
    """
    print(f"  Evaluating {len(df)} samples for model '{model_name}'...") # Added verbosity
    setups = df['setup'].tolist()
    gt_punchlines = df['ground_truth_punchline'].tolist()
    # Handle potential missing values or type issues gracefully
    # generated = df['generated_punchline'].fillna('').astype(str).apply(lambda x: x.split('\n')[0]).tolist()
    generated = df['generated_punchline'].fillna('').astype(str).apply(lambda x: ' '.join(x.split()[:15])).tolist()
    # generated = df['generated_punchline'].fillna('').astype(str).tolist()

    references = [f"{i} {j}" for i, j in zip(setups, gt_punchlines)]
    predictions = generated
    
    # Calculate average length of generated content
    # For Mistral models: only punchline length
    # For other models: full generated content length (setup + punchline)
    if "Mistral Zero-Shot" in model_name or 'Mistral Finetuned' in model_name:
        print("  Applying different evaluation (including setup in prediction).") # Added verbosity
        predictions = [f"{i} {j}" for i, j in zip(setups, generated)]
        # For Mistral models, average length is just the punchline
        avg_length = np.mean([len(g.split()) for g in generated])
    else:
        # For other models, generated_punchline contains setup+punchline, so measure full length
        avg_length = np.mean([len(g.split()) for g in generated])

    # --- Compute Metrics ---
    print("    Computing BLEU...") # Added verbosity
    bleu_res = bleu_metric.compute(predictions=predictions, references=references)
    print("    Computing ROUGE...") # Added verbosity
    rouge_res = rouge_metric.compute(predictions=predictions, references=references)
    print("    Computing BERTScore...") # Added verbosity
    # Consider adding batch_size parameter if memory issues arise or for potential speedup
    bert_res = bertscore_metric.compute(predictions=predictions, references=references, lang="en", batch_size=128, verbose=True)
    # bert_res = bertscore_metric.compute(predictions=predictions, references=references, lang="en")
    print("    Evaluation complete for this batch.") # Added verbosity
    # --- End Compute Metrics ---

    return {
        'bleu': bleu_res['bleu'],
        'rougeL': rouge_res['rougeL'],
        'bertscore': np.mean(bert_res['f1']), # Ensure we get a single mean value
        'avg_length': avg_length
    }


def process_seeds(seed_files, model_name):
    """
    Process and compute metrics for all seeds of a model.
    """
    metrics = {'bleu': [], 'rougeL': [], 'bertscore': [], 'avg_length': []}
    num_seeds = len(seed_files)

    print(f"Processing {num_seeds} seed files for model '{model_name}':") # Added verbosity
    for i, seed_file in enumerate(seed_files):
        print(f"  Processing seed file {i+1}/{num_seeds}: {seed_file}") # Added verbosity
        try:
            df = pd.read_csv(seed_file)
            # Basic check for required columns
            required_cols = ['setup', 'ground_truth_punchline', 'generated_punchline']
            if not all(col in df.columns for col in required_cols):
                print(f"    Warning: Skipping file {seed_file} due to missing columns. Required: {required_cols}")
                continue # Skip this file if columns are missing

            # Pass the specific model name for potential logic inside evaluate_model
            res = evaluate_model(df, model_name)
            metrics['bleu'].append(res['bleu'])
            metrics['rougeL'].append(res['rougeL'])
            metrics['bertscore'].append(res['bertscore'])
            metrics['avg_length'].append(res['avg_length'])
            print(f"    Finished processing {seed_file}.") # Added verbosity

        except FileNotFoundError:
            print(f"    Warning: Seed file not found: {seed_file}. Skipping.") # Added warning
        except Exception as e:
            print(f"    Warning: Error processing file {seed_file}: {e}. Skipping.") # Added generic error handling

    if not metrics['bleu']: # Check if any seeds were successfully processed
         print(f"  Warning: No valid data processed for model '{model_name}'. Returning zero stats.")
         stats = {
             'bleu_mean': 0.0, 'bleu_std': 0.0,
             'rougeL_mean': 0.0, 'rougeL_std': 0.0,
             'bertscore_mean': 0.0, 'bertscore_std': 0.0,
             'avg_length_mean': 0.0, 'avg_length_std': 0.0,
         }
    else:
        # Calculate stats only if there's data
        print(f"  Calculating statistics for model '{model_name}'...") # Added verbosity
        stats = {
            'bleu_mean': np.mean(metrics['bleu']),
            'bleu_std': np.std(metrics['bleu']),
            'rougeL_mean': np.mean(metrics['rougeL']),
            'rougeL_std': np.std(metrics['rougeL']),
            'bertscore_mean': np.mean(metrics['bertscore']),
            'bertscore_std': np.std(metrics['bertscore']),
            'avg_length_mean': np.mean(metrics['avg_length']),
            'avg_length_std': np.std(metrics['avg_length']),
        }
        print(f"  Statistics calculated for {model_name}.") # Added verbosity

    return stats


# Define seed files for each model
# Make sure these paths are correct relative to where you run the script
models = {
    'Finetuned GPT-2': [f"../results/unfiltered_finetuned_gpt2_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD Imitation+DPO': [f"../results/unfiltered_gpt2_acl_dpo_seed_{i}.csv" for i in range(5)],
    'Mistral Zero-Shot': [f"../results/filtered_zero_shot_mistral_seed_{i}.csv" for i in range(5)],
    'Mistral Finetuned': [f"../results/unfiltered_finetuned_mistral_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Finetuned Mistral)': [f"../results/unfiltered_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(5)],
    'GPT-2 Mistral (LoRA)': [f"../results/unfiltered_gpt2_mistral_lora_seed_{i}.csv" for i in range(5)],
    'GPT-2 Mistral (Frozen)': [f"../results/unfiltered_frozen_mistral_seed_{i}.csv" for i in range(5)], #this is gpt2_frozen_mistral
    'GPT-2 KD (Imitation)': [f"../results/unfiltered_gpt2_imitation_seed_{i}.csv" for i in range(5)],
}


all_results = []
print("\n--- Starting Evaluation Loop for All Models ---") # Added verbosity
total_models = len(models)

for i, (name, files) in enumerate(models.items()):
    print(f"\n--- Processing Model {i+1}/{total_models}: {name} ---") # Added verbosity
    start_model_time = time.time()
    stats = process_seeds(files, name)
    all_results.append({
        'Model': name,
        'BLEU': f"{stats['bleu_mean']:.4f} ± {stats['bleu_std']:.4f}",
        'ROUGE-L': f"{stats['rougeL_mean']:.4f} ± {stats['rougeL_std']:.4f}",
        'BERTScore': f"{stats['bertscore_mean']:.4f} ± {stats['bertscore_std']:.4f}",
        'Avg Length': f"{stats['avg_length_mean']:.1f} ± {stats['avg_length_std']:.1f}",
    })
    end_model_time = time.time()
    print(f"--- Finished Model {name} in {end_model_time - start_model_time:.2f} seconds ---") # Added verbosity

print("\n--- All Models Processed. Generating Final Results Table ---") # Added verbosity

# Check if any results were generated
if not all_results:
    print("No results were generated. Exiting.")
else:
    # Print the final table
    print(tabulate(all_results, headers="keys", tablefmt="grid"))

print("\n--- Evaluation Script Finished ---") # Added verbosity

In [ ]:
import pandas as pd
import evaluate
import numpy as np
from tabulate import tabulate
import time # Import time for timing

# --- Load evaluation metrics (assuming already loaded from previous script if run sequentially) ---
# If running this script independently, uncomment these lines and add loading messages.
# print("Loading evaluation metrics...")
# start_time = time.time()
# bleu_metric = evaluate.load("bleu")
# rouge_metric = evaluate.load("rouge")
# bertscore_metric = evaluate.load("bertscore")
# print(f"Metrics loaded in {time.time() - start_time:.2f} seconds.")
# --- End loading metrics ---


def compute_comparative_metrics(df, finetuned_df, model_name, finetuned_model_name="Finetuned GPT-2", include_setup_in_prediction=False):
    """
    Compute BLEU, ROUGE-1/2/L/Lsum, and BERTScore between predictions and finetuned references.
    Includes verbosity.
    """
    print(f"    Computing comparative metrics for {len(df)} samples ({model_name} vs {finetuned_model_name})...") # Added verbosity

    # Prepare texts, handle potential NaN/type issues
    setups = df['setup'].fillna('').astype(str).tolist()
    # preds_punchlines = df['generated_punchline'].fillna('').astype(str).apply(lambda x: x.split('\n')[0]).tolist()
    # refs_punchlines = finetuned_df['generated_punchline'].fillna('').astype(str).apply(lambda x: x.split('\n')[0]).tolist()
    preds_punchlines = df['generated_punchline'].fillna('').astype(str).tolist()
    refs_punchlines = finetuned_df['generated_punchline'].fillna('').astype(str).tolist()
    
    # refs_setups = finetuned_df['setup'].fillna('').astype(str).tolist() # Needed if include_setup_in_prediction is True

    # Combine setup and punchline for predictions
    # preds = [f"{s} {p}" for s, p in zip(setups, preds_punchlines)]
    preds = preds_punchlines
    refs = refs_punchlines

    # # Combine setup and punchline for references *only if* requested
    if "Zero" in model_name:
        print("      Including setup in reference texts for comparison.") # Added verbosity
        preds = [f"{s} {p}" for s, p in zip(setups, preds_punchlines)]
    # else:
    #     # Default: Compare generated (setup+punchline) vs reference (punchline only)
    #     # Or consider if comparison should always be setup+punchline vs setup+punchline?
    #     # Current implementation compares setup+pred_punchline vs ref_punchline (unless include_setup_in_prediction=True)
    #     # Let's adjust refs to also include setup for a more direct comparison of full generations
    #     print("      Comparing full generation (Setup + Pred Punchline) vs (Setup + Ref Punchline).") # Clarification
    #     refs = [f"{s} {p}" for s, p in zip(refs_setups, refs_punchlines)]
    #     # If you strictly want to compare against ONLY the reference punchline:
    #     # refs = refs_punchlines
    #     # print("      Comparing full generation (Setup + Pred Punchline) vs (Ref Punchline Only).")


    # --- Compute Metrics ---
    print("      Computing BLEU...") # Added verbosity
    bleu = bleu_metric.compute(predictions=preds, references=refs)['bleu']

    print("      Computing ROUGE...") # Added verbosity
    rouge = rouge_metric.compute(
        predictions=preds,
        references=refs,
        use_stemmer=True,
        rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"]
    )

    print("      Computing BERTScore...") # Added verbosity
    # Consider adding batch_size parameter if memory issues arise or for potential speedup
    bert = bertscore_metric.compute(predictions=preds, references=refs, lang="en", batch_size=128, verbose=True)
    bert_f1 = np.mean(bert['f1']) # Ensure we get a single mean value
    print("    Comparative metrics computation complete for this pair.") # Added verbosity
    # --- End Compute Metrics ---

    return {
        'bleu': bleu,
        'rouge1': rouge['rouge1'],
        'rouge2': rouge['rouge2'],
        'rougeL': rouge['rougeL'],
        'rougeLsum': rouge['rougeLsum'],
        'bertscore': bert_f1
    }


def process_model_comparisons(seed_files, finetuned_files, model_name, include_setup=False):
    """
    Process comparative metrics across seeds for one model versus fine-tuned GPT-2.
    Includes verbosity and error handling.
    """
    metrics = {'bleu': [], 'rouge1': [], 'rouge2': [], 'rougeL': [], 'rougeLsum': [], 'bertscore': []}
    num_seeds = len(seed_files)
    finetuned_model_name = "Finetuned GPT-2" # Assuming this is constant for comparisons

    print(f"Processing {num_seeds} seed comparisons for model '{model_name}' vs '{finetuned_model_name}':") # Added verbosity

    processed_count = 0
    for i, (samp_file, finetuned_file) in enumerate(zip(seed_files, finetuned_files)):
        print(f"  Comparing seed pair {i+1}/{num_seeds}:") # Added verbosity
        print(f"    Model file: {samp_file}")
        print(f"    Reference file: {finetuned_file}")
        try:
            df_model = pd.read_csv(samp_file)
            df_finetuned = pd.read_csv(finetuned_file)

            # Basic check for required columns
            required_cols_model = ['setup', 'generated_punchline']
            required_cols_ref = ['setup', 'generated_punchline'] # Need setup from ref if include_setup=True or for consistent comparison
            if not all(col in df_model.columns for col in required_cols_model):
                print(f"    Warning: Skipping model file {samp_file} due to missing columns. Required: {required_cols_model}")
                continue
            if not all(col in df_finetuned.columns for col in required_cols_ref):
                print(f"    Warning: Skipping reference file {finetuned_file} due to missing columns. Required: {required_cols_ref}")
                continue
            if len(df_model) != len(df_finetuned):
                 print(f"    Warning: Skipping pair due to mismatched lengths ({len(df_model)} vs {len(df_finetuned)}). Ensure files correspond.")
                 continue


            # Pass model names for clearer logging inside compute_comparative_metrics
            res = compute_comparative_metrics(df_model, df_finetuned, model_name, finetuned_model_name, include_setup)
            for k, v in res.items():
                metrics[k].append(v)
            processed_count += 1
            print(f"    Finished comparison for seed pair {i+1}.") # Added verbosity

        except FileNotFoundError as e:
            print(f"    Warning: File not found: {e}. Skipping this seed pair.") # Added specific warning
        except Exception as e:
            print(f"    Warning: Error processing file pair ({samp_file}, {finetuned_file}): {e}. Skipping.") # Added generic error handling

    if processed_count == 0: # Check if any seeds were successfully processed
         print(f"  Warning: No valid data processed for comparison '{model_name}' vs '{finetuned_model_name}'. Returning zero stats.")
         stats = {
             'Model': model_name,
             'BLEU': "0.0000 ± 0.0000", 'ROUGE-1': "0.0000 ± 0.0000",
             'ROUGE-2': "0.0000 ± 0.0000", 'ROUGE-L': "0.0000 ± 0.0000",
             'ROUGE-Lsum': "0.0000 ± 0.0000", 'BERTScore': "0.0000 ± 0.0000"
         }
    else:
        # Aggregate and format results only if there's data
        print(f"  Aggregating statistics over {processed_count} processed seed pairs for '{model_name}'...") # Added verbosity
        stats = {
            'Model': model_name,
            'BLEU': f"{np.mean(metrics['bleu']):.4f} ± {np.std(metrics['bleu']):.4f}",
            'ROUGE-1': f"{np.mean(metrics['rouge1']):.4f} ± {np.std(metrics['rouge1']):.4f}",
            'ROUGE-2': f"{np.mean(metrics['rouge2']):.4f} ± {np.std(metrics['rouge2']):.4f}",
            'ROUGE-L': f"{np.mean(metrics['rougeL']):.4f} ± {np.std(metrics['rougeL']):.4f}",
            'ROUGE-Lsum': f"{np.mean(metrics['rougeLsum']):.4f} ± {np.std(metrics['rougeLsum']):.4f}",
            'BERTScore': f"{np.mean(metrics['bertscore']):.4f} ± {np.std(metrics['bertscore']):.4f}"
        }
        print(f"  Statistics aggregated for {model_name}.") # Added verbosity
    return stats


# --- Main Execution Block ---

finetuned_gpt2_files = [f"../results/unfiltered_finetuned_gpt2_seed_{i}.csv" for i in range(5)]
comparisons = {
    'GPT-2 KD Imitation+DPO': [f"../results/unfiltered_gpt2_acl_dpo_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Finetuned Mistral)': [f"../results/unfiltered_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in range(5)],
    # 'GPT-2 Mistral (LoRA)': [f"../results/gpt2_mistral_seed_{i}.csv" for i in range(1)],
    'GPT-2 Mistral (Frozen)': [f"../results/unfiltered_frozen_mistral_seed_{i}.csv" for i in range(5)],
    'GPT-2 KD (Imitation)': [f"../results/unfiltered_gpt2_imitation_seed_{i}.csv" for i in range(5)],
}

all_comparison_results = []
print("\n--- Starting Comparative Evaluation Loop ---") # Added verbosity
total_comparisons = len(comparisons)

for i, (name, files) in enumerate(comparisons.items()):
    print(f"\n--- Comparing Model {i+1}/{total_comparisons}: {name} vs Finetuned GPT-2 ---") # Added verbosity
    start_comp_time = time.time()

    # Check if model name suggests BART-style (setup inclusion) - adjust if needed
    # For this specific comparison logic, we decided *always* to compare setup+pred vs setup+ref
    # So `include_setup` here primarily controls if the *reference* text should include setup for metrics.
    # Let's keep it consistent: compare full generated text (setup+punchline) vs full reference text (setup+punchline).
    # The `include_setup` flag becomes less critical here, but we pass False unless explicitly needed.
    include_setup_flag = 'BART' in name # Example: remains False for current models
    if include_setup_flag:
         print(f"  Note: '{name}' identified as potentially BART-style. Flag 'include_setup_in_prediction' set to True (though comparison logic might override).")


    stats = process_model_comparisons(files, finetuned_gpt2_files, name, include_setup=include_setup_flag)
    all_comparison_results.append(stats)

    end_comp_time = time.time()
    print(f"--- Finished comparison for {name} in {end_comp_time - start_comp_time:.2f} seconds ---") # Added verbosity

print("\n--- All Model Comparisons Processed. Generating Final Results Table ---") # Added verbosity

# Check if any results were generated
if not all_comparison_results:
    print("No comparison results were generated. Exiting.")
else:
    # Display table
    print("\nComparative Metrics (vs Finetuned GPT-2):") # Added table title
    print(tabulate(all_comparison_results, headers="keys", tablefmt="grid"))

print("\n--- Comparative Evaluation Script Finished ---") # Added verbosity

In [ ]:
import pandas as pd
import evaluate
import numpy as np
from tabulate import tabulate
import time # Import time for timing

# --- Load evaluation metrics (assuming already loaded from previous script if run sequentially) ---
# If running this script independently, uncomment these lines and add loading messages.
# print("Loading evaluation metrics...")
# start_time = time.time()
# bleu_metric = evaluate.load("bleu")
# rouge_metric = evaluate.load("rouge")
# bertscore_metric = evaluate.load("bertscore")
# print(f"Metrics loaded in {time.time() - start_time:.2f} seconds.")
# --- End loading metrics ---


def compute_comparative_metrics(df, finetuned_df, model_name, finetuned_model_name="Finetuned Mistral", include_setup_in_prediction=False):
    """
    Compute BLEU, ROUGE-1/2/L/Lsum, and BERTScore between predictions and finetuned references.
    Includes verbosity.
    """
    print(f"    Computing comparative metrics for {len(df)} samples ({model_name} vs {finetuned_model_name})...") # Added verbosity

    # Prepare texts, handle potential NaN/type issues
    setups = df['setup'].fillna('').astype(str).tolist()
    # preds_punchlines = df['generated_punchline'].fillna('').astype(str).apply(lambda x: x.split('\n')[0]).tolist()
    # refs_punchlines = finetuned_df['generated_punchline'].fillna('').astype(str).apply(lambda x: x.split('\n')[0]).tolist()
    preds_punchlines = df['generated_punchline'].fillna('').astype(str).tolist()
    refs_punchlines = finetuned_df['generated_punchline'].fillna('').astype(str).tolist()
    
    # refs_setups = finetuned_df['setup'].fillna('').astype(str).tolist() # Needed if include_setup_in_prediction is True

    # Combine setup and punchline for predictions
    # preds = [f"{s} {p}" for s, p in zip(setups, preds_punchlines)]
    preds = preds_punchlines
    refs = refs_punchlines

    # # Combine setup and punchline for references *only if* requested
    if "Zero" in model_name:
        print("      Including setup in reference texts for comparison.") # Added verbosity
        preds = [f"{s} {p}" for s, p in zip(setups, preds_punchlines)]
    # else:
    #     # Default: Compare generated (setup+punchline) vs reference (punchline only)
    #     # Or consider if comparison should always be setup+punchline vs setup+punchline?
    #     # Current implementation compares setup+pred_punchline vs ref_punchline (unless include_setup_in_prediction=True)
    #     # Let's adjust refs to also include setup for a more direct comparison of full generations
    #     print("      Comparing full generation (Setup + Pred Punchline) vs (Setup + Ref Punchline).") # Clarification
    #     refs = [f"{s} {p}" for s, p in zip(refs_setups, refs_punchlines)]
    #     # If you strictly want to compare against ONLY the reference punchline:
    #     # refs = refs_punchlines
    #     # print("      Comparing full generation (Setup + Pred Punchline) vs (Ref Punchline Only).")


    # --- Compute Metrics ---
    print("      Computing BLEU...") # Added verbosity
    bleu = bleu_metric.compute(predictions=preds, references=refs)['bleu']

    print("      Computing ROUGE...") # Added verbosity
    rouge = rouge_metric.compute(
        predictions=preds,
        references=refs,
        use_stemmer=True,
        rouge_types=["rouge1", "rouge2", "rougeL", "rougeLsum"]
    )

    print("      Computing BERTScore...") # Added verbosity
    # Consider adding batch_size parameter if memory issues arise or for potential speedup
    bert = bertscore_metric.compute(predictions=preds, references=refs, lang="en", batch_size=128, verbose=True)
    bert_f1 = np.mean(bert['f1']) # Ensure we get a single mean value
    print("    Comparative metrics computation complete for this pair.") # Added verbosity
    # --- End Compute Metrics ---

    return {
        'bleu': bleu,
        'rouge1': rouge['rouge1'],
        'rouge2': rouge['rouge2'],
        'rougeL': rouge['rougeL'],
        'rougeLsum': rouge['rougeLsum'],
        'bertscore': bert_f1
    }


def process_model_comparisons(seed_files, finetuned_files, model_name, include_setup=False):
    """
    Process comparative metrics across seeds for one model versus fine-tuned Mistral.
    Includes verbosity and error handling.
    """
    metrics = {'bleu': [], 'rouge1': [], 'rouge2': [], 'rougeL': [], 'rougeLsum': [], 'bertscore': []}
    num_seeds = len(seed_files)
    finetuned_model_name = "Finetuned Mistral" # Assuming this is constant for comparisons

    print(f"Processing {num_seeds} seed comparisons for model '{model_name}' vs '{finetuned_model_name}':") # Added verbosity

    processed_count = 0
    for i, (samp_file, finetuned_file) in enumerate(zip(seed_files, finetuned_files)):
        print(f"  Comparing seed pair {i+1}/{num_seeds}:") # Added verbosity
        print(f"    Model file: {samp_file}")
        print(f"    Reference file: {finetuned_file}")
        try:
            df_model = pd.read_csv(samp_file)
            df_finetuned = pd.read_csv(finetuned_file)

            # Basic check for required columns
            required_cols_model = ['setup', 'generated_punchline']
            required_cols_ref = ['setup', 'generated_punchline'] # Need setup from ref if include_setup=True or for consistent comparison
            if not all(col in df_model.columns for col in required_cols_model):
                print(f"    Warning: Skipping model file {samp_file} due to missing columns. Required: {required_cols_model}")
                continue
            if not all(col in df_finetuned.columns for col in required_cols_ref):
                print(f"    Warning: Skipping reference file {finetuned_file} due to missing columns. Required: {required_cols_ref}")
                continue
            if len(df_model) != len(df_finetuned):
                 print(f"    Warning: Skipping pair due to mismatched lengths ({len(df_model)} vs {len(df_finetuned)}). Ensure files correspond.")
                 continue


            # Pass model names for clearer logging inside compute_comparative_metrics
            res = compute_comparative_metrics(df_model, df_finetuned, model_name, finetuned_model_name, include_setup)
            for k, v in res.items():
                metrics[k].append(v)
            processed_count += 1
            print(f"    Finished comparison for seed pair {i+1}.") # Added verbosity

        except FileNotFoundError as e:
            print(f"    Warning: File not found: {e}. Skipping this seed pair.") # Added specific warning
        except Exception as e:
            print(f"    Warning: Error processing file pair ({samp_file}, {finetuned_file}): {e}. Skipping.") # Added generic error handling

    if processed_count == 0: # Check if any seeds were successfully processed
         print(f"  Warning: No valid data processed for comparison '{model_name}' vs '{finetuned_model_name}'. Returning zero stats.")
         stats = {
             'Model': model_name,
             'BLEU': "0.0000 ± 0.0000", 'ROUGE-1': "0.0000 ± 0.0000",
             'ROUGE-2': "0.0000 ± 0.0000", 'ROUGE-L': "0.0000 ± 0.0000",
             'ROUGE-Lsum': "0.0000 ± 0.0000", 'BERTScore': "0.0000 ± 0.0000"
         }
    else:
        # Aggregate and format results only if there's data
        print(f"  Aggregating statistics over {processed_count} processed seed pairs for '{model_name}'...") # Added verbosity
        stats = {
            'Model': model_name,
            'BLEU': f"{np.mean(metrics['bleu']):.4f} ± {np.std(metrics['bleu']):.4f}",
            'ROUGE-1': f"{np.mean(metrics['rouge1']):.4f} ± {np.std(metrics['rouge1']):.4f}",
            'ROUGE-2': f"{np.mean(metrics['rouge2']):.4f} ± {np.std(metrics['rouge2']):.4f}",
            'ROUGE-L': f"{np.mean(metrics['rougeL']):.4f} ± {np.std(metrics['rougeL']):.4f}",
            'ROUGE-Lsum': f"{np.mean(metrics['rougeLsum']):.4f} ± {np.std(metrics['rougeLsum']):.4f}",
            'BERTScore': f"{np.mean(metrics['bertscore']):.4f} ± {np.std(metrics['bertscore']):.4f}"
        }
        print(f"  Statistics aggregated for {model_name}.") # Added verbosity
    return stats


# --- Main Execution Block ---

finetuned_mistral_files = [f"../results/unfiltered_finetuned_mistral_seed_{i}.csv" for i in [2, 3, 4]]
comparisons = {
    'Finetuned GPT-2': [f"../results/unfiltered_finetuned_gpt2_seed_{i}.csv" for i in [2, 3, 4]],
    'GPT-2 KD Imitation+DPO': [f"../results/unfiltered_gpt2_acl_dpo_seed_{i}.csv" for i in [2, 3, 4]],
    'Mistral Zero-Shot': [f"../results/filtered_zero_shot_mistral_seed_{i}.csv" for i in [2, 3, 4]],
    'GPT-2 KD (Finetuned Mistral)': [f"../results/unfiltered_gpt2_finetuned_mistral_kd_seed_{i}.csv" for i in [2, 3, 4]],
    'GPT-2 Mistral (LoRA)': [f"../results/unfiltered_gpt2_mistral_lora_seed_{i}.csv" for i in [2, 3, 4]],
    'GPT-2 Mistral (Frozen)': [f"../results/unfiltered_frozen_mistral_seed_{i}.csv" for i in [2, 3, 4]],
    'GPT-2 KD (Imitation)': [f"../results/unfiltered_gpt2_imitation_seed_{i}.csv" for i in [2, 3, 4]],
}

all_comparison_results = []
print("\n--- Starting Comparative Evaluation Loop ---") # Added verbosity
total_comparisons = len(comparisons)

for i, (name, files) in enumerate(comparisons.items()):
    print(f"\n--- Comparing Model {i+1}/{total_comparisons}: {name} vs Finetuned Mistral ---") # Added verbosity
    start_comp_time = time.time()

    # Check if model name suggests BART-style (setup inclusion) - adjust if needed
    # For this specific comparison logic, we decided *always* to compare setup+pred vs setup+ref
    # So `include_setup` here primarily controls if the *reference* text should include setup for metrics.
    # Let's keep it consistent: compare full generated text (setup+punchline) vs full reference text (setup+punchline).
    # The `include_setup` flag becomes less critical here, but we pass False unless explicitly needed.
    include_setup_flag = 'BART' in name # Example: remains False for current models
    if include_setup_flag:
         print(f"  Note: '{name}' identified as potentially BART-style. Flag 'include_setup_in_prediction' set to True (though comparison logic might override).")


    stats = process_model_comparisons(files, finetuned_mistral_files, name, include_setup=include_setup_flag)
    all_comparison_results.append(stats)

    end_comp_time = time.time()
    print(f"--- Finished comparison for {name} in {end_comp_time - start_comp_time:.2f} seconds ---") # Added verbosity

print("\n--- All Model Comparisons Processed. Generating Final Results Table ---") # Added verbosity

# Check if any results were generated
if not all_comparison_results:
    print("No comparison results were generated. Exiting.")
else:
    # Display table
    print("\nComparative Metrics (vs Finetuned Mistral):") # Added table title
    print(tabulate(all_comparison_results, headers="keys", tablefmt="grid"))

print("\n--- Comparative Evaluation Script Finished ---") # Added verbosity